# train_upload_hive（简版）

`train_upload_hive.ipynb` 的简化版：**Hive 连接、取数、依赖安装**的逻辑都收进了 `train_upload_hive_lib.py`（使用时与本文件放在 ModelArts 的**同一目录**），notebook 只留配置与训练/上传流程。完整逐步排障版见 `train_upload_hive.ipynb`。

## 1. 配置区 ⚠️ 必须修改

> 把下方 `OBS_BUCKET` 改成你自己的桶名；换区域时同步改 `OBS_ENDPOINT`。
> Notebook 绑定了 OBS 委托的话，AK/SK 留空即可（moxing 自动使用委托）；
> 没有委托就临时填自己的 AK/SK——**用完即清空，绝不要把真实密钥提交进仓库或分享出去**。


In [ ]:
# ==================== ⚠️ 必须修改 ====================
OBS_BUCKET   = "<你的桶名>"                             # 你的 OBS 桶名
OBS_PREFIX   = "models"                                # OBS 存储路径前缀（= 服务端 OBS_KEY 的前半段）
OBS_ENDPOINT = "obs.cn-north-4.myhuaweicloud.com"      # OBS endpoint（换区域同步改，部署侧 OBS_ENDPOINT 也要改）

# IAM 访问密钥（华为云控制台 → 我的凭证 → 访问密钥）
# ⚠️ Notebook 绑定了 OBS 委托就留空（moxing 自动用委托）；没有委托才临时填自己的
# ⚠️ 真实 AK/SK 用完即清空，绝不要提交进仓库或分享出去
ACCESS_KEY_ID     = ""    # ← 填你的 AK（或留空用委托）
SECRET_ACCESS_KEY = ""    # ← 填你的 SK（或留空用委托）
# ==========================================================

import os
from pathlib import Path

# 工作目录：ModelArts Notebook 用自带可写目录，本地运行自动落到当前目录
WORK_DIR = "/home/ma-user/work/xgb_train" if Path("/home/ma-user").exists() else "."

assert "<" not in OBS_BUCKET, "请先把 <你的桶名> 换成实际桶名"

WORK = Path(WORK_DIR)
WORK.mkdir(parents=True, exist_ok=True)
OLD_DIR = WORK / "model_out" / "old"
NEW_DIR = WORK / "model_out" / "new"
OLD_DIR.mkdir(parents=True, exist_ok=True)
NEW_DIR.mkdir(parents=True, exist_ok=True)

# 本地保留两套模型（用于对比验证）
OLD_MODEL_LOCAL = OLD_DIR / "xgboost_breast_cancer.json"
NEW_MODEL_LOCAL = NEW_DIR / "xgboost_breast_cancer.json"

# OBS 只有一个目标路径（不分 old/new 子目录，靠 §7 的 ACTIVE_MODEL 切换）
ACTIVE_MODEL_OBS = f"obs://{OBS_BUCKET}/{OBS_PREFIX}/xgboost_breast_cancer.json"

print(f"工作目录:      {WORK}")
print(f"OBS 桶:        {OBS_BUCKET}")
print(f"OBS 目标路径:  {ACTIVE_MODEL_OBS}")

# --- MRS Hive 连接配置（实测值，单一事实来源 hive_export/MRS_RUN.md §0；换集群改这里）---
HIVE_HOST = "10.0.0.15"    # HiveServer2 内网 IP（master1）
HIVE_PORT = 21066          # HiveServer2 Thrift 端口
DATABASE  = "default"
USERNAME  = "hhx"          # MRS 业务用户（kinit 时输它的密码）
REALM     = "252A63EC_2C90_4B5A_B4D7_17A3077B1CB8.COM"
KDC_HOSTS = ["10.0.0.15", "10.0.0.51"]  # 两个 Master 都写，容错
KDC_PORT  = 21732          # 华为 MRS 专用 KDC 端口，不是 88！
print(f"Hive: {HIVE_HOST}:{HIVE_PORT}  用户: {USERNAME}@{REALM}")


## 2. 准备依赖

`train_upload_hive_lib.py` 的 `ensure_training_deps()` 缺啥装啥（pandas / scikit-learn / xgboost），安装过程**带心跳+实时进度**；已装则直接跳过。`from ... import *` 的那一刻已自动完成 ModelArts 的 pandas ABI 修复（sys.path 摘除 modelarts-sdk）。


In [ ]:
from train_upload_hive_lib import *

ensure_training_deps()   # 缺啥装啥（心跳+进度）；ModelArts 通常已预装，秒过

import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score
from xgboost import XGBClassifier

print(f"pandas {pd.__version__} <- {pd.__file__}")
print(f"numpy  {np.__version__}")

# ModelArts 自带的 OBS 操作库（moxing 认证依赖 §1 的 AK/SK，因此留在 notebook）
try:
    import moxing as mox
    if ACCESS_KEY_ID and SECRET_ACCESS_KEY:
        import moxing.framework.content_db as content_db
        content_db.configure_obs_credentials(
            access_key_id=ACCESS_KEY_ID,
            secret_access_key=SECRET_ACCESS_KEY,
            endpoint=OBS_ENDPOINT,
        )
        print("已用 AK/SK 配置 moxing 认证")
    else:
        print("未提供 AK/SK，使用 Notebook 委托认证")
    HAS_MOXING = True
except ImportError:
    HAS_MOXING = False
    print("moxing 不可用，将尝试 esdk-obs-python 作为 fallback")


## 3. 连接 MRS Hive（Kerberos 安全集群）

一行完成：网络探测 → 环境预装（kinit/cyrus-sasl/pyhive，**带心跳进度**）→ krb5.conf → kinit（弹密码框输 `hhx` 的密码）→ 连接 HiveServer2。默认值即实测值；逐步排障与原理见 `modelarts_hive_conn.ipynb`、`MRS_RUN.md`、ADR-0002。

> 全程幂等：**票据 24h 过期或内核重启后，重跑本格即可**（已装的依赖、已有的票据自动跳过）。


In [ ]:
conn = connect_mrs_hive(
    hive_host=HIVE_HOST, hive_port=HIVE_PORT, database=DATABASE,
    username=USERNAME, realm=REALM, kdc_hosts=KDC_HOSTS, kdc_port=KDC_PORT,
)


## 4. 从 Hive 加载数据 + 样本定义

`fetch_breast_cancer(conn)`（lib 里）负责小块取数（每次 5 行，绕开部分环境 libsasl2 的大帧解密 bug）。本格把 Hive 下划线列名还原成 sklearn 带空格特征名，做数据校验，并嵌入 `sample_request.json` 的测试样本。


In [ ]:
# === 从 Hive 读取（小块取数在 lib 里，绕开 libsasl2 大帧 bug）===
df_hive = fetch_breast_cancer(conn)

# Hive 列名是下划线风格（mean_radius），改回 sklearn 的带空格风格（"mean radius"），
# 与 sample_request.json / app.py 的特征名保持一致
df_hive.columns = [c.replace("_", " ") for c in df_hive.columns]

X = df_hive.drop(columns=["target"])
y = df_hive["target"].astype(int)
print(f"Hive breast_cancer: {X.shape[0]} 样本, {X.shape[1]} 特征")

# === 数据校验 ===
# 特征名只借自 sklearn（不用它的数据），保证与推理服务的特征顺序严格一致
from sklearn.datasets import load_breast_cancer
FEATURE_NAMES = list(load_breast_cancer().feature_names)
assert list(X.columns) == FEATURE_NAMES, f"列名与 sklearn 特征名不一致: {list(X.columns)[:3]} ..."
assert X.shape == (569, 30), f"期望 569x30, 实际 {X.shape}"
assert set(y.unique()) <= {0, 1}, f"target 取值异常: {sorted(y.unique())}"
print("[OK] 校验通过：569x30、特征名与 sklearn 一致、target ∈ {0,1}")

# === 嵌入测试样本（来自 sample_request.json）===
sample_row = {
    "mean radius": 17.99, "mean texture": 10.38, "mean perimeter": 122.8,
    "mean area": 1001.0, "mean smoothness": 0.1184, "mean compactness": 0.2776,
    "mean concavity": 0.3001, "mean concave points": 0.1471,
    "mean symmetry": 0.2419, "mean fractal dimension": 0.07871,
    "radius error": 1.095, "texture error": 0.9053, "perimeter error": 8.589,
    "area error": 153.4, "smoothness error": 0.006399,
    "compactness error": 0.04904, "concavity error": 0.05373,
    "concave points error": 0.01587, "symmetry error": 0.03003,
    "fractal dimension error": 0.006193, "worst radius": 25.38,
    "worst texture": 17.33, "worst perimeter": 184.6, "worst area": 2019.0,
    "worst smoothness": 0.1622, "worst compactness": 0.6656,
    "worst concavity": 0.7119, "worst concave points": 0.2654,
    "worst symmetry": 0.4601, "worst fractal dimension": 0.1189,
}
sample_df = pd.DataFrame([sample_row], columns=FEATURE_NAMES)
print(f"测试样本: {len(sample_row)} 特征")


## 5. 训练函数

训练 → 评估 → 保存到本地 → 打印样本预测值。
新旧模型对同一样本预测值不同，这正是后面热切换验证的判定依据。


In [ ]:
def train_and_save(params, random_state, output_path, label):
    """训练模型，评估并保存，返回样本预测概率。"""
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=random_state, stratify=y,
    )
    model = XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        tree_method="hist",
        n_jobs=-1,
        random_state=random_state,
        **params,
    )
    model.fit(X_train, y_train, verbose=False)

    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    acc = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba)
    pred = float(model.predict_proba(sample_df)[0, 1])

    output_path.parent.mkdir(parents=True, exist_ok=True)
    model.save_model(str(output_path))
    size = output_path.stat().st_size

    print(f"\n{'='*55}")
    print(f"  {label}")
    print(f"{'='*55}")
    print(f"  超参:     {params}")
    print(f"  Accuracy: {acc:.4f}  AUC: {auc:.4f}")
    print(f"  样本预测: {pred:.16f}")
    print(f"  本地保存: {output_path} ({size:,} bytes)")
    return pred


## 6. 训练 OLD 模型（baseline）

100 棵树，浅深度，高学习率。


In [ ]:
old_pred = train_and_save(
    params=dict(
        n_estimators=100, max_depth=3, learning_rate=0.1,
        subsample=0.8, colsample_bytree=0.8,
    ),
    random_state=42,
    output_path=OLD_MODEL_LOCAL,
    label="OLD MODEL (baseline)",
)


## 7. 训练 NEW 模型（不同超参）

250 棵树，深深度，低学习率，加正则化。


In [ ]:
new_pred = train_and_save(
    params=dict(
        n_estimators=250, max_depth=6, learning_rate=0.01,
        subsample=0.6, colsample_bytree=0.5,
        min_child_weight=5, reg_alpha=0.5, reg_lambda=2.0, gamma=0.5,
    ),
    random_state=2024,
    output_path=NEW_MODEL_LOCAL,
    label="NEW MODEL (updated)",
)


## 8. 上传模型到 OBS 🚀

把训练好的模型上传到 OBS 的**单一目标路径**（推理服务只认这个路径）：

- `obs://{OBS_BUCKET}/{OBS_PREFIX}/xgboost_breast_cancer.json`

> 推理服务（`app.py`，环境变量 `OBS_BUCKET` + `OBS_KEY`）从该路径读取模型。
> 想切换 old / new：改下方 `ACTIVE_MODEL` 后重跑本 cell 即可。


In [ ]:
# 选择当前要上传哪套模型到 OBS（改成 "new" 可切换为新模型）
ACTIVE_MODEL = "old"   # "old" 或 "new"

LOCAL_TO_UPLOAD = OLD_MODEL_LOCAL if ACTIVE_MODEL == "old" else NEW_MODEL_LOCAL
print(f"当前选择: {ACTIVE_MODEL} 模型")
print(f"本地文件: {LOCAL_TO_UPLOAD} ({LOCAL_TO_UPLOAD.stat().st_size:,} bytes)")
print(f"OBS 目标: {ACTIVE_MODEL_OBS}")
print()

def upload_to_obs(local_path, obs_uri, label=""):
    """上传单个文件到 OBS（覆盖写）。"""
    tag = f" [{label}]" if label else ""
    print(f"  上传{tag}: {local_path} → {obs_uri}")

    if HAS_MOXING:
        mox.file.copy(str(local_path), obs_uri)
    else:
        # fallback: esdk-obs-python
        from obs import ObsClient
        assert ACCESS_KEY_ID and SECRET_ACCESS_KEY, "AK/SK 必填（moxing 不可用时）"
        client = ObsClient(
            access_key_id=ACCESS_KEY_ID,
            secret_access_key=SECRET_ACCESS_KEY,
            server=f"https://{OBS_ENDPOINT}",
        )
        key = obs_uri.replace(f"obs://{OBS_BUCKET}/", "")
        resp = client.putFile(OBS_BUCKET, key, str(local_path))
        assert resp.status < 300, f"上传失败: status={resp.status}"
        client.close()

    size = local_path.stat().st_size
    print(f"    ✅ 完成 ({size:,} bytes)")

upload_to_obs(LOCAL_TO_UPLOAD, ACTIVE_MODEL_OBS, ACTIVE_MODEL.upper())
print(f"\n🚀 上传完成！")
print(f"   推理服务 app.py 会从 {ACTIVE_MODEL_OBS} 读取模型。")
print(f"   切换模型：把 ACTIVE_MODEL 改成 'new'，重跑这个 cell 即可。")
